In [1]:
import pandas as pd
import numpy as np

import pocket as pk
import vc_analysis as vc


occupancy_data = pd.read_csv('datatraining.txt', header=0)
occupancy_data = occupancy_data.rename(columns = {'Occupancy':'Classification'})
del occupancy_data['date']
# we need to convert these digits 1 and 5 into a binary signal [1,-1]
occupancy_data['Classification'] = occupancy_data['Classification'].apply(lambda value: value if value==1 else -1)


result = pk.pocket(occupancy_data, iterations=30)
count_missclassified = vc.misclassified_count(w=result['w'], data=occupancy_data)
vc_bound = vc.vc_bound(N=10, tolerance=.07, mH=3)

print count_missclassified
print vc_bound

{'misclassified': 558.0, 'ratio_misclassified': 0.06852511359449834}
2.49531144611


## The VC_dimension

Before we actually train our classification alogrithm on our dataset, let's take a look at what the VC dimension can tell us about what we're about to do. 

---
**Theorem 2.5** (VC generalization bound).  For any $ \delta > 0,$

$$ E_{out}(g) \le E_{in}(g) + \sqrt{ \frac{8}{N} ln \frac{4mH(2N)}{\delta}} $$

with probability $\ge 1 - \delta $

---

What we are saying is that in order to know how well our algorithm will generalize to data that it has not seen($E_{out}$), we must frame it in terms of $E_{in}$ + the result of our VC calcuation. 

The important part is that our VC calculation is dependent on $N$(amount of data we train our algorithm with), and $\delta$(percentage error we are willing to tolerate). 

As an example, let us say that our $E_{in}$ is *.06* and the result of our VC calculation is *.04*, where our error tolerance $\delta$ was *.03*.

We would then be able to say

$$ E_{out} \le .06 + .04 with probability \ge 1 - .03 $$

This in effect says with 97% certainty, $E_{out}$ will be less than .10%. So we are 93% certain that our algorithm will missclassify no more than 10% of samples that it has not seen. 

Powerful stuff, but does it work? 
